In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from web3 import Web3
import requests
import sys
sys.path.append('/Users/noahturner/code/Blockchain-Project/oracle')
from odds_client import get_client

load_dotenv(find_dotenv())

contract_address = os.getenv("CONTRACT_ADDRESS")
rpc_url = os.getenv("RPC_URL")
etherscan_api_key = os.getenv("ETHERSCAN_API_KEY")
chain_id = os.getenv("NEXT_PUBLIC_CHAIN_ID")
odds_api_key = os.getenv("SPORTS_API_KEY")
bettor = "0x244e18F228b2804f3a4445d2446c1847b55B294A"
# getenv returns str | None; function also accepts int or str
first_block = int(os.getenv("CONTRACT_DEPLOY_BLOCK") or "0")






In [5]:

# def get_raw_logs():
#     contract_address = os.getenv("CONTRACT_ADDRESS")
#     rpc_url = os.getenv("RPC_URL")
#     etherscan_api_key = os.getenv("ETHERSCAN_API_KEY")
chain_id = os.getenv("NEXT_PUBLIC_CHAIN_ID")
resp = requests.get(
    f"https://api.etherscan.io/v2/api?chainid={chain_id}&module=proxy&action=eth_blockNumber&apikey={etherscan_api_key}"
).json()
latest_block_hex = resp["result"]      # e.g. "0xa2d1f3"
latest_block = int(latest_block_hex, 16)
response = requests.get(f"https://api.etherscan.io/v2/api?chainid={chain_id}&module=logs&action=getLogs&address={contract_address}&fromBlock={first_block}&toBlock={latest_block}&page=1&offset=100&apikey={etherscan_api_key}")

logs = response.json()['result']


len(logs)

11

In [6]:
import json
with open('contract_abi.json', 'r') as f:
    abi = json.load(f)

abi = abi['abi']

w3 = Web3(Web3.HTTPProvider(rpc_url))  # you can also use Web3() just for decode

c = w3.eth.contract(address=Web3.to_checksum_address(contract_address), abi=abi)

In [7]:
def bets_for_bettor(logs, c, bettor: str):

    decoded_bets = []
    bets = []
    for l in logs:
        if not l.get("topics"):
            continue
        try:
            evt = c.events.BetPlaced().process_log({
                "address": Web3.to_checksum_address(l["address"]),
                "topics": l["topics"],
                "data": l["data"],
                "blockNumber": int(l["blockNumber"], 16),
                "transactionHash": l["transactionHash"],
                "transactionIndex": int(l["transactionIndex"], 16),
                "blockHash": l["blockHash"],
                "logIndex": int(l["logIndex"], 16),
            })
            decoded_bets.append(dict(evt["args"]))
        except Exception:
            # not BetPlaced (or decode mismatch), skip
            pass
    for bet in decoded_bets:
        if bet["bettor"] == bettor:
            bets.append(bet)
    return decoded_bets
bets_for_bettor(logs, c, bettor)

[{'betId': 0,
  'bettor': '0x244e18F228b2804f3a4445d2446c1847b55B294A',
  'gameId': '70506078',
  'betType': 0,
  'outcome': 1,
  'line': 0,
  'amount': 50000000000000000,
  'odds': 216},
 {'betId': 1,
  'bettor': '0x244e18F228b2804f3a4445d2446c1847b55B294A',
  'gameId': '70506090',
  'betType': 0,
  'outcome': 1,
  'line': 0,
  'amount': 1000000000000000,
  'odds': 300},
 {'betId': 2,
  'bettor': '0x244e18F228b2804f3a4445d2446c1847b55B294A',
  'gameId': '70506094',
  'betType': 0,
  'outcome': 0,
  'line': 0,
  'amount': 1000000000000000,
  'odds': 252},
 {'betId': 3,
  'bettor': '0x513F545750D134985c3F212282DFEef663f05CfF',
  'gameId': '62079212',
  'betType': 0,
  'outcome': 0,
  'line': 0,
  'amount': 6900000000000000,
  'odds': 143},
 {'betId': 4,
  'bettor': '0x513F545750D134985c3F212282DFEef663f05CfF',
  'gameId': '70506094',
  'betType': 0,
  'outcome': 0,
  'line': 0,
  'amount': 3000000000000000,
  'odds': 246}]

In [22]:
with get_client() as client:
    try:
        event = client.get_event_by_id('62079212')
    except Exception as e:
        if str(e) == 'Resource not found':
            print('yep')
event

{'id': 62079212,
 'home': 'Colorado Avalanche',
 'away': 'Seattle Kraken',
 'homeId': 3682,
 'awayId': 794340,
 'date': '2026-04-17T02:00:00Z',
 'sport': {'name': 'Ice Hockey', 'slug': 'ice-hockey'},
 'league': {'name': 'USA - NHL', 'slug': 'usa-nhl'},
 'status': 'live',
 'scores': {'home': 1, 'away': 0, 'periods': {'p1': {'home': 0, 'away': 0}}}}